# 02 — Feature Parity Audit (Section 5.3)

The most common silent failure in a deployed ML system: features computed at training time (full-history batch) drift from features computed at request time (trailing window). Same (symbol, date), different value, no error raised — the served predictions just quietly degrade.

This notebook runs `features/feature_parity.py` against the **real** feature pipeline across the universe and reports any feature whose value depends on how much history it was given. On synthetic data this already caught `obv` and `vwap_cumulative` (both cumulative-from-inception sums). This is the audit that must pass before serving.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

In [2]:
# Ensure the project root is on the import path so `src` can be imported.
project_root = Path.cwd().resolve()
for candidate in (project_root, *project_root.parents):
    if (candidate / "src").exists():
        sys.path.insert(0, str(candidate))
        break

from src.forecast_engine.data.loader import RAW_PRICES_PATH, load_raw_prices
from src.forecast_engine.features.indicators import add_technical_indicators
from src.forecast_engine.features.schema import FEATURE_NAMES
from src.forecast_engine.features import feature_parity as fp

raw = load_raw_prices()
raw["date"] = pd.to_datetime(raw["date"])
raw = raw.sort_values(["symbol", "date"]).reset_index(drop=True)

print(f"{len(raw):,} raw rows, {raw['symbol'].nunique()} tickers")
print(f"Dates: {raw['date'].min().date()} to {raw['date'].max().date()}")
print(f'Source: {RAW_PRICES_PATH}')
raw.head()

622,731 raw rows, 502 tickers
Dates: 2021-07-19 to 2026-07-17
Source: D:\StockForecastRisk\data\raw\sp500_yfinance_daily.parquet


,date,symbol,security,gics_sector,gics_sub_industry,adj_close,close,high,low,open,volume
0,2021-07-19,A,Agilent Technologies,Health Care,Life Sciences Tools & Services,142.434052,147.580002,147.970001,146.949997,147.660004,1835100.0
1,2021-07-20,A,Agilent Technologies,Health Care,Life Sciences Tools & Services,144.094101,149.300003,151.250000,147.830002,148.449997,2246000.0
2,2021-07-21,A,Agilent Technologies,Health Care,Life Sciences Tools & Services,143.756302,148.949997,149.910004,147.600006,149.490005,2255700.0
3,2021-07-22,A,Agilent Technologies,Health Care,Life Sciences Tools & Services,145.059174,150.300003,150.500000,148.669998,149.710007,1972800.0
4,2021-07-23,A,Agilent Technologies,Health Care,Life Sciences Tools & Services,146.999084,152.309998,152.410004,150.429993,150.589996,2132000.0


## Compute both paths and compare

For a sample of tickers: compute features over full history (training path), then recompute over trailing windows ending at several recent dates (serving path), and compare the shared rows. Any mismatch is a feature whose value is history-length-dependent — a parity bug.

In [3]:
SAMPLE_TICKERS = list(raw["symbol"].unique()[:20])
OFFSETS = [1, 5, 20, 50]   # recent dates a request might ask for

all_mismatches = []
for symbol in SAMPLE_TICKERS:
    history = raw[raw["symbol"] == symbol].reset_index(drop=True)
    if len(history) < fp.SERVE_WARMUP + max(OFFSETS):
        continue
    training = add_technical_indicators(history)
    for offset in OFFSETS:
        as_of = len(history) - offset
        window = fp.serving_window(history, as_of_index=as_of)
        serving = add_technical_indicators(window)
        as_of_date = history["date"].iloc[as_of]
        cols = [c for c in FEATURE_NAMES if c in training.columns and c in serving.columns]
        tr = training[training["date"] == as_of_date]
        sv = serving[serving["date"] == as_of_date]
        if tr.empty or sv.empty:
            continue
        mm = fp.parity_mismatches(tr, sv, cols)
        all_mismatches.append(mm)

mismatches = pd.concat(all_mismatches, ignore_index=True) if all_mismatches else pd.DataFrame()
print(f"total (row, feature) mismatches: {len(mismatches)}")

total (row, feature) mismatches: 96


## ★ Which features break parity?

In [4]:
if mismatches.empty:
    print("PARITY HOLDS: every feature is history-length-independent. Safe to serve.")
else:
    offenders = mismatches["feature"].value_counts()
    print("Features that BREAK train/serve parity (must fix before serving):\n")
    print(offenders.to_string())
    print("\nExample mismatches:")
    display(mismatches.head(10))
    print("\nThese features produce different values depending on how much history")
    print("was fed. Cumulative-from-inception features (obv, vwap_cumulative) are the")
    print("usual cause. Fix: drop the raw level, keep the differenced/rolling version")
    print("(e.g. obv_change_5), then re-run this notebook -- it should report zero.")

Features that BREAK train/serve parity (must fix before serving):

feature
obv            80
macd_signal     7
macd            5
macd_hist       4

Example mismatches:


,symbol,date,feature,train_value,serve_value
0,A,2026-07-17,macd_hist,8.228016e-02,8.228029e-02
1,A,2026-07-17,obv,4.129320e+07,3.909760e+07
2,A,2026-07-13,obv,4.162790e+07,4.438580e+07
3,A,2026-06-18,obv,2.834090e+07,2.948410e+07
4,A,2026-05-06,obv,7.434200e+06,1.230340e+07
5,AAPL,2026-07-17,obv,3.297593e+09,1.318684e+09
6,AAPL,2026-07-13,obv,3.166744e+09,1.173675e+09
7,AAPL,2026-06-18,obv,2.885387e+09,1.076930e+09
8,AAPL,2026-05-06,obv,2.754775e+09,1.157462e+09
9,ABBV,2026-07-17,obv,4.062033e+08,-4.535000e+05



These features produce different values depending on how much history
was fed. Cumulative-from-inception features (obv, vwap_cumulative) are the
usual cause. Fix: drop the raw level, keep the differenced/rolling version
(e.g. obv_change_5), then re-run this notebook -- it should report zero.


## Conclusions

**Zero mismatches** — the feature pipeline is safe to serve; training and serving compute identical values for every date.

**Any mismatch** — those features are history-length-dependent and will silently corrupt served predictions. The fix is structural: remove the cumulative-from-inception *level* and keep only a differenced or rolling-window form, which depends solely on recent history and so matches the served window. Re-run until this reports zero, then add `tests/test_feature_parity.py` to CI so a future change can never reintroduce the drift.

This audit is cheap and catches a failure class that never appears in training metrics — the training path is internally consistent, so R² looks fine while served predictions are wrong.